# Itens 06 / 07 / 08 — a escada de comparação treinada

Marco pós-qualificação. Este notebook produz os **18 treinos** que sustentam a comparação
única do item 10: três modelos neurais × dois regimes de split × três seeds.

| variante | o que muda no grafo | ticket |
|---|---|---|
| `proposta` | nada, é o MusicDiffusionGNN sobre o grafo do [ADR-0003](../docs/adr/0003-atributos-de-genero-derivados.md) | [06](../.scratch/next-milestone/issues/06-gnn-retreinada-nova-matriz.md) |
| `sem_grafo` | **todas** as arestas esvaziadas: o `SAGEConv` cai no ramo raiz e o encoder vira um MLP por nó | [07](../.scratch/next-milestone/issues/07-baseline-neural-sem-grafo.md) |
| `embaralhado` | arestas religadas ao acaso preservando grau e semana de estreia; só a topologia muda | [08](../.scratch/next-milestone/issues/08-gnn-grafo-embaralhado.md) |

As três compartilham features de nó, arquitetura, orçamento de parâmetros, protocolo de split e
orçamento de treino. É isso que torna a diferença entre elas atribuível ao que se quis isolar:
`proposta` − `sem_grafo` mede capacidade neural contra estrutura, `proposta` − `embaralhado` mede
**topologia** sem o confundidor de trocar de arquitetura ([ADR-0002](../docs/adr/0002-grafo-embaralhado-como-controle.md)).

O pré-compromisso do que a dissertação passa a afirmar em cada desfecho está em
[ADR-0001](../docs/adr/0001-precompromisso-de-falseamento.md), registrado antes de qualquer
número. Este notebook não lê desfecho nenhum: ele só treina. A leitura é o item 10.

**Custo e retomada.** Um treino levou 21,4 min na T4 (item 05, seed 42/`current`). Dezoito ficam
em torno de 6,5 h, acima do que uma sessão do Colab costuma durar. Cada treino grava seu
checkpoint no Drive assim que termina e o laço pula o que já existe: rode o notebook quantas
vezes forem necessárias, ele continua de onde parou. `TEMPO_LIMITE_MIN` corta o laço antes da
desconexão para não perder um treino pela metade.


## 0. Ambiente — Colab (GPU) ou local

No Colab, clona o repositório (código + artefatos de dados versionados) e instala o PyG.
Ative a GPU em *Ambiente de execução → Alterar tipo de runtime → GPU*.
Repo privado: cole um PAT em `GITHUB_TOKEN`.

> Os grafos `hetero_full_current.pt` e `hetero_full_pre_pandemia.pt` precisam estar **commitados**
> antes de clonar: os CSVs brutos da rede de gêneros não são versionados, então o grafo não pode
> ser reconstruído aqui dentro.


In [ ]:
import sys, os, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except Exception:
    IS_COLAB = False

REPO_URL     = "https://github.com/cristianomendieta/music-influence-gnn.git"
REPO_BRANCH  = "main"
REPO_DIR     = "/content/music-influence-gnn"
GITHUB_TOKEN = ""   # repo privado: cole um PAT aqui OU defina a env GITHUB_TOKEN

DATA_FILES = [
    "data/processed/graph/hetero_full_current.pt",
    "data/processed/graph/hetero_full_pre_pandemia.pt",
    "data/processed/graph/node_id_map.json",
    "data/processed/timeseries.parquet",
]

def _clone(url):
    return subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, url, REPO_DIR])

if IS_COLAB:
    if Path(REPO_DIR, "pyproject.toml").exists():
        # O runtime sobrevive ao restart do kernel: um clone de sessão anterior ficaria
        # para trás em silêncio e o import viria do código velho. Sincroniza sempre.
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{REPO_BRANCH}"], check=True)
    else:
        tok = GITHUB_TOKEN or os.environ.get("GITHUB_TOKEN", "")
        url = REPO_URL.replace("https://", f"https://{tok}@") if tok else REPO_URL
        print(f"Clonando {REPO_URL} (branch {REPO_BRANCH})...")
        if _clone(url).returncode != 0:
            from getpass import getpass
            tok = getpass("Clone falhou (repo privado?). Cole um GitHub token (PAT): ")
            _clone(REPO_URL.replace("https://", f"https://{tok}@")).check_returncode()
    os.chdir(REPO_DIR)
    print("commit em uso:", subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"],
                                           capture_output=True, text=True).stdout.strip())
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch-geometric", "pyarrow"], check=True)
    faltando = [f for f in DATA_FILES if not Path(REPO_DIR, f).exists()]
    print("✓ Colab pronto. cwd =", os.getcwd())
    if faltando:
        print("⚠️ FALTAM no repositório:", faltando)
else:
    print("Local — nada a clonar.")


In [ ]:
import json, time, shutil
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

_anchor = Path(globals().get("__vsc_ipynb_file__", os.getcwd()))
if not _anchor.exists():
    _anchor = Path(os.getcwd())
ROOT = _anchor if _anchor.is_dir() else _anchor.parent
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), f"raiz do projeto não encontrada a partir de {_anchor}"
sys.path.insert(0, str(ROOT / "src"))

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 10
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

torch.manual_seed(42); np.random.seed(42)

from music_diffusion_gnn.graph.build import graph_path
from music_diffusion_gnn.graph.controls import (
    edge_report, rewire_preserving_degree, strip_all_edges,
)
from music_diffusion_gnn.models.diffusion_gnn import MusicDiffusionGNN
from music_diffusion_gnn.evaluation.stats import aggregate_seeds
from music_diffusion_gnn.training.dataset import (
    aggregate_weekly, build_pop_bank, build_samples, temporal_split,
)
from music_diffusion_gnn.training.trainer import Config, train_one

GRAPH_DIR = ROOT / "data" / "processed" / "graph"
NMAP_PATH = GRAPH_DIR / "node_id_map.json"
TS_PATH   = ROOT / "data" / "processed" / "timeseries.parquet"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if IS_COLAB and DEVICE != "cuda":
    raise RuntimeError("GPU não ativada no Colab: Ambiente de execução → Alterar tipo de runtime → GPU.")
print("ROOT =", ROOT, "| DEVICE =", DEVICE)


In [ ]:
# Artefatos DURÁVEIS no Drive (sobrevivem à desconexão do Colab)
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/music-influence-gnn/escada_treinos")
else:
    OUT = ROOT / "results" / "escada_treinos"
OUT.mkdir(parents=True, exist_ok=True)
print("artefatos →", OUT)


## 1. A matriz de treinos

`W=12, hidden=128, layers=3, lr=5e-4` é a config vencedora da grid da qualificação, a mesma
re-treinada no item 05 sobre o grafo reconstruído. Ela fica **fixa** nas três variantes: o objeto
de estudo aqui é a estrutura, não o hiperparâmetro.

`MAX_COTRAJ_TREINO = 30_000` é o teto de arestas de cotrajetória por snapshot que o treino sempre
usou (`Config.max_cotraj_edges`), imposto originalmente por memória de autograd. Ele fica
registrado em cada checkpoint porque o item 08 exige que o orçamento de arestas do treino seja
comparável ao da avaliação: a avaliação (notebook do item 09) roda as **duas** leituras, grafo
completo e mesmo teto de 30k, e reporta as duas.

A ordem do plano é por regime e por seed, com as três variantes juntas: assim que a seed 42 de
`current` fecha, já existe uma escada completa para olhar, mesmo que a sessão caia depois.


In [ ]:
# ------------------------------- configuração ------------------------------- #
W, HIDDEN, LAYERS, LR = 12, 128, 3, 5e-4   # config vencedora da qualificação
SEEDS      = [42, 43, 44]
REGIMES    = ["current", "pre_pandemia"]
VARIANTES  = ["proposta", "sem_grafo", "embaralhado"]
MAX_COTRAJ_TREINO = 30_000                 # teto de arestas de cotrajetória por snapshot
TEMPO_LIMITE_MIN  = 150                    # corta o laço antes da sessão do Colab cair

PLANO = [(v, r, s) for r in REGIMES for s in SEEDS for v in VARIANTES]

def variante_grafo(g, variante: str, seed: int):
    """O grafo que cada variante enxerga. Nenhuma das funções muta `g`."""
    if variante == "proposta":
        return g
    if variante == "sem_grafo":
        return strip_all_edges(g)
    if variante == "embaralhado":
        # A seed do religamento é a seed do experimento: cada réplica vê uma
        # topologia aleatória diferente, então a dispersão entre seeds do controle
        # já inclui a variação do próprio embaralhamento, não só a do treino.
        return rewire_preserving_degree(g, seed=seed)
    raise ValueError(variante)

print(f"{len(PLANO)} treinos no plano:")
for i, p in enumerate(PLANO, 1):
    print(f"  {i:2d}. {p[0]:<12s} {p[1]:<13s} seed {p[2]}")


## 2. Dados por regime

As amostras dependem só do regime (mesmo `W`, mesmo mapa de nós), então são construídas uma vez e
reaproveitadas pelas três variantes. `first_seen` é calculado sobre a série **completa**: se saísse
de dentro do split, a primeira semana de cada split viraria falsamente a estreia da música e a
janela sairia errada.


In [ ]:
ts = pd.read_parquet(TS_PATH)
weekly = aggregate_weekly(ts)
first_seen = weekly.groupby(["song_id", "chart"], observed=True)["week"].min().to_dict()

_cache_dados: dict[str, dict] = {}

def dados(regime: str) -> dict:
    if regime not in _cache_dados:
        g = torch.load(graph_path(regime, GRAPH_DIR), weights_only=False)
        splits_df = temporal_split(weekly, regime=regime)
        pop_bank = build_pop_bank(weekly, NMAP_PATH, n_music=g["music"].num_nodes)
        samples = {
            nome: build_samples(splits_df[nome], W=W, node_id_map_path=NMAP_PATH,
                                first_seen=first_seen)
            for nome in ("train", "val", "test")
        }
        print(f"{regime}: " + " ".join(f"{k}={len(v):,}" for k, v in samples.items())
              + f" | train_years={list(g['genre'].train_years)}")
        _cache_dados[regime] = {"g": g, "splits_df": splits_df, "pop_bank": pop_bank,
                                "samples": samples}
    return _cache_dados[regime]

_ = dados("current")


## 3. Os controles são o que dizem ser

Antes de gastar GPU: as duas transformações do grafo precisam preservar exatamente o que
prometem. `strip_all_edges` tem que zerar **todas** as arestas (o item 07 exige teste de que o
modelo não acessa relação nenhuma) e `rewire_preserving_degree` tem que manter contagem por tipo,
grau de saída e grau de entrada, mudando só quem se liga a quem.

A suíte `tests/test_graph_controls.py` já garante isso no CI; a célula abaixo repete a checagem
sobre o grafo real desta execução, que é o que de fato entra no treino.


In [ ]:
if IS_COLAB:
    subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_graph_controls.py"], check=True)

g_ref = dados("current")["g"]
g_sem = strip_all_edges(g_ref)
g_emb = rewire_preserving_degree(g_ref, seed=42)

rel = pd.concat({
    "real": edge_report(g_ref).set_index("edge_type"),
    "embaralhado": edge_report(g_emb).set_index("edge_type"),
    "sem_grafo": edge_report(g_sem).set_index("edge_type"),
}, axis=1)
display(rel[[("real", "n_edges"), ("embaralhado", "n_edges"), ("sem_grafo", "n_edges"),
             ("real", "grau_saida_max"), ("embaralhado", "grau_saida_max"),
             ("real", "grau_entrada_max"), ("embaralhado", "grau_entrada_max")]])

# sem_grafo: zero arestas em todo tipo
assert sum(g_sem[et].edge_index.shape[1] for et in g_sem.edge_types) == 0

# embaralhado: contagem e graus idênticos, topologia diferente
identicas = {}
for et in g_ref.edge_types:
    a, b = g_ref[et].edge_index, g_emb[et].edge_index
    assert a.shape == b.shape
    assert torch.equal(a[0], b[0])                       # mesmos nós de origem, mesma ordem
    assert torch.equal(torch.bincount(a[1], minlength=g_ref[et[2]].num_nodes),
                       torch.bincount(b[1], minlength=g_ref[et[2]].num_nodes))  # grau de entrada
    identicas[str(et)] = float((a[1] == b[1]).float().mean())
print("\nfração de arestas que caíram no mesmo destino após o religamento:")
for k, v in identicas.items():
    print(f"  {k:<45s} {v:.4f}")


## 4. Orçamento de parâmetros

O item 07 pede o número declarado: se o baseline sem grafo tivesse menos capacidade, um empate
seria trivialmente explicado pela capacidade e não pela ausência de estrutura. Aqui as três
variantes instanciam a **mesma** classe com a mesma configuração, e o grafo só muda quais
mensagens trafegam. O `SAGEConv` tem parâmetros lazy, então o forward de uma semana é o que
materializa a contagem.


In [ ]:
d = dados("current")
linhas = []
for variante in VARIANTES:
    m = MusicDiffusionGNN(d["g"].metadata(), hidden=HIDDEN, layers=LAYERS, dropout=0.2,
                          pop_bank=d["pop_bank"]).to(DEVICE)
    with torch.no_grad():
        m.encode_weeks(variante_grafo(d["g"], variante, 42).to(DEVICE), [0],
                       max_cotraj_edges=MAX_COTRAJ_TREINO)
    linhas.append({"variante": variante, "n_params": m.count_params()})
    del m
if DEVICE == "cuda":
    torch.cuda.empty_cache()
params = pd.DataFrame(linhas)
display(params)
assert params.n_params.nunique() == 1, "as variantes deixaram de ter o mesmo orçamento de parâmetros"
print(f"orçamento único: {params.n_params.iloc[0]:,} parâmetros nas três variantes")


## 5. O laço de treino

Cada treino grava o checkpoint no Drive assim que termina, com a variante, o regime, a seed, o
teto de arestas e a contagem de parâmetros dentro do arquivo: o item 06 exige que o artefato
registre o que o gerou.

O checkpoint da seed 42 de `current` já existe desde o item 05, com o nome antigo
`gnn_current_seed42.pt`. A primeira célula o adota sob o nome novo em vez de re-treinar: são o
mesmo grafo, a mesma config e a mesma seed.

Rode esta célula quantas vezes precisar. Ela sempre continua de onde parou.


In [ ]:
t_inicio = time.time()
for variante, regime, seed in PLANO:
    ckpt   = OUT / f"gnn_{variante}_{regime}_seed{seed}.pt"
    legado = OUT / f"gnn_{regime}_seed{seed}.pt"     # nome usado pelo notebook do item 05
    if variante == "proposta" and legado.exists() and not ckpt.exists():
        shutil.copy(legado, ckpt)
        print(f"{variante}/{regime}/seed{seed}: adotado do item 05 ({legado.name})")

    if ckpt.exists():
        r = torch.load(ckpt, map_location="cpu", weights_only=False)
        print(f"{variante}/{regime}/seed{seed}: pronto (val_mse={r['val_mse']:.6f}) — pulando")
        continue

    decorrido = (time.time() - t_inicio) / 60
    if decorrido > TEMPO_LIMITE_MIN:
        print(f"\nlimite de {TEMPO_LIMITE_MIN} min atingido ({decorrido:.0f} min). "
              "Rode a célula de novo numa sessão nova: os checkpoints prontos serão pulados.")
        break

    d = dados(regime)
    g_var = variante_grafo(d["g"], variante, seed)
    cfg = Config(W=W, hidden=HIDDEN, layers=LAYERS, lr=LR, seed=seed,
                 max_cotraj_edges=MAX_COTRAJ_TREINO)
    print(f"\n▶ {variante}/{regime}/seed{seed} — {cfg}")
    r = train_one(cfg, {"train": d["samples"]["train"], "val": d["samples"]["val"]},
                  g_var, device=DEVICE, pop_bank=d["pop_bank"])
    torch.save({
        "variante": variante, "split_regime": regime, "seed": seed,
        "graph_seed": seed if variante == "embaralhado" else None,
        "config_str": str(cfg), "W": cfg.W, "hidden": cfg.hidden, "layers": cfg.layers,
        "lr": cfg.lr, "dropout": cfg.dropout, "max_cotraj_edges": cfg.max_cotraj_edges,
        "n_params": r.n_params, "val_mse": r.val_mse,
        "val_curve": r.val_curve, "train_curve": r.train_curve,
        "state_dict": {k: v.cpu() for k, v in r.best_state_dict.items()},
    }, ckpt)
    print(f"  val_mse={r.val_mse:.6f}  params={r.n_params:,}  t={r.elapsed_sec/60:.1f}min → {ckpt.name}")
    del g_var
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print("\nfeitos:", sum(1 for v, r_, s in PLANO if (OUT / f"gnn_{v}_{r_}_seed{s}.pt").exists()),
      f"de {len(PLANO)}")


## 6. Onde a matriz está

`val_mse` é a métrica de parada do treino, não a comparação do marco: ela é de um passo, sobre a
validação, na leitura completa. A comparação que decide o pré-compromisso são os três horizontes
nos dois recortes, e é o item 10 quem a faz. Esta tabela serve para duas coisas: ver o que falta
treinar, e ter a primeira indicação de ordem entre as variantes.


In [ ]:
linhas = []
for variante, regime, seed in PLANO:
    ck = OUT / f"gnn_{variante}_{regime}_seed{seed}.pt"
    if not ck.exists():
        linhas.append({"variante": variante, "regime": regime, "seed": seed, "val_mse": np.nan})
        continue
    r = torch.load(ck, map_location="cpu", weights_only=False)
    linhas.append({"variante": variante, "regime": regime, "seed": seed,
                   "val_mse": r["val_mse"], "n_params": r.get("n_params")})
matriz = pd.DataFrame(linhas)
display(matriz.pivot_table(index=["regime", "seed"], columns="variante", values="val_mse"))

agg = []
for (variante, regime), grp in matriz.dropna(subset=["val_mse"]).groupby(["variante", "regime"]):
    media, desvio = aggregate_seeds(grp["val_mse"].to_numpy())
    agg.append({"variante": variante, "regime": regime, "n_seeds": len(grp),
                "val_mse_medio": media, "val_mse_desvio": desvio})
agg = pd.DataFrame(agg).sort_values(["regime", "val_mse_medio"])
display(agg)
matriz.to_parquet(OUT / "matriz_val_mse.parquet", index=False)


In [ ]:
fig, axes = plt.subplots(1, len(REGIMES), figsize=(6 * len(REGIMES), 4.2), squeeze=False)
cores = {"proposta": "#2a78d6", "sem_grafo": "#888888", "embaralhado": "#eb6834"}
for ax, regime in zip(axes[0], REGIMES):
    for variante in VARIANTES:
        for seed in SEEDS:
            ck = OUT / f"gnn_{variante}_{regime}_seed{seed}.pt"
            if not ck.exists():
                continue
            curva = torch.load(ck, map_location="cpu", weights_only=False).get("val_curve")
            if curva:
                ax.plot(curva, color=cores[variante], alpha=0.75,
                        label=variante if seed == SEEDS[0] else None)
    ax.set_title(regime); ax.set_xlabel("época"); ax.set_ylabel("val MSE"); ax.set_yscale("log")
    ax.legend()
fig.tight_layout(); plt.show()


## 7. O que levar de volta

Do Drive (`escada_treinos/`):

- `gnn_{variante}_{regime}_seed{seed}.pt` — os 18 checkpoints, cada um carregando dentro de si a
  variante, o regime, a seed, o teto de arestas do treino e a contagem de parâmetros;
- `matriz_val_mse.parquet` — a grade de `val_mse`, insumo do texto de método.

Em seguida, `item09_avaliacao_matriz_colab.ipynb`: ele lê estes checkpoints, roda os três
horizontes nos dois recortes e nas duas leituras de orçamento de arestas, e refaz a ablação por
tipo de aresta com o harness corrigido. Só depois vem `item10_comparacao_consolidada.ipynb`, que
junta SIR e persistência e aciona o pré-compromisso.
